# Terratorch - Use a FM for Burned area delineation task

In [ ]:
! pip install terratorch==1.1
! pip install numpy==1.26.4 --force-reinstall

## Available models in TerraTorch

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="numpy.dtype size changed")
warnings.filterwarnings("ignore", message="numpy.ufunc size changed")

from terratorch import BACKBONE_REGISTRY
print("Available backbone models:")
for model_name in BACKBONE_REGISTRY:
    print(f"- {model_name}")

## Customization of existing Dataset in TorchGeo : ChabuD

ChaBuD is a dataset for Burned area delineation from Sentinel-2 images.
We need to modify the original dataset to return only post fire images and only a subset of bands because the model has been trained on Sentinel-2 images with only B, G, R, Narrow NIR, and SWIR bands, corresponding to indexes 1,2,3,7, 10, 11


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.figure import Figure
from torch import Tensor
from torchgeo.datasets.utils import percentile_normalization
from torchgeo.datasets import ChaBuD

class CustomChaBuDDataset(ChaBuD):
    def __init__(self, temporal: str,  *args, **kwargs):
        super().__init__(*args, **kwargs)
        assert temporal in ["pre", "post", "all"], f"Invalid temporal value: {temporal}. Must be one of ['pre', 'post', 'all']"
        self.temporal = temporal
        self.out_bands = len(self.bands)

    def __getitem__(self, index):
        sample = super().__getitem__(index)
        # print(sample["image"].shape, sample["mask"].shape)
        if self.temporal == "pre":
            # return only the first half of the sample
            sample["image"] = sample["image"][:self.out_bands, :, :]

        elif self.temporal == "post":
            # return only the second half of the sample
            sample["image"] = sample["image"][self.out_bands:, :, :]
        # if self.temporal == "all", return the full sample
        return sample

    def plot(
        self,
        sample: dict[str, Tensor],
        show_titles: bool = True,
        suptitle: str | None = None,
    ) -> Figure:
        """Plot a sample from the dataset.

        Args:
            sample: a sample returned by :meth:`__getitem__`
            show_titles: flag indicating whether to show titles above each panel
            suptitle: optional suptitle to use for figure

        Returns:
            a matplotlib Figure with the rendered sample
        """
        rgb_indices = []
        for band in self.rgb_bands:
            if band in self.bands:
                # print(f"Band {band} found in dataset bands.")
                rgb_indices.append(self.bands.index(band))
            else:
                raise ValueError(
                    "Dataset doesn't contain some of the RGB bands")
        mask = sample['mask'].numpy()
        image_pre = sample['image'][rgb_indices].numpy()
        image_post = sample['image'][rgb_indices].numpy()
        image_pre = percentile_normalization(image_pre)
        image_post = percentile_normalization(image_post)

        ncols = 3

        showing_predictions = 'prediction' in sample
        if showing_predictions:
            prediction = sample['prediction']
            ncols += 1

        fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(10, ncols * 5))

        axs[0].imshow(np.transpose(image_pre, (1, 2, 0)))
        axs[0].axis('off')
        axs[1].imshow(np.transpose(image_post, (1, 2, 0)))
        axs[1].axis('off')
        axs[2].imshow(mask)
        axs[2].axis('off')

        if showing_predictions:
            axs[3].imshow(prediction)
            axs[3].axis('off')

        if show_titles:
            axs[0].set_title('Image Pre')
            axs[1].set_title('Image Post')
            axs[2].set_title('Mask')
            if showing_predictions:
                axs[3].set_title('Prediction')

        if suptitle is not None:
            plt.suptitle(suptitle)

        return fig


In [ ]:

from typing import Any
from torchgeo.datamodules.geo import NonGeoDataModule
import torch
from einops import repeat

class CustomChaBuDDataModule(NonGeoDataModule):
    """LightningDataModule implementation for the ChaBuD dataset.

    Uses the train/val splits from the dataset

    .. versionadded:: 0.6
    """

    # min/max values computed on train set using 2/98 percentiles
    min = torch.tensor(
        [0.0, 1.0, 73.0, 39.0, 46.0, 25.0, 26.0, 21.0, 17.0, 1.0, 20.0, 21.0]
    )
    max = torch.tensor(
        [
            1926.0,
            2174.0,
            2527.0,
            2950.0,
            3237.0,
            3717.0,
            4087.0,
            4271.0,
            4290.0,
            4219.0,
            4568.0,
            3753.0,
        ]
    )

    def __init__(
        self, temporal: str = "all", batch_size: int = 64, num_workers: int = 0, **kwargs: Any
    ) -> None:
        """Initialize a new ChaBuDDataModule instance.

        Args:
            batch_size: Size of each mini-batch.
            num_workers: Number of workers for parallel data loading.
            **kwargs: Additional keyword arguments passed to
                :class:`~torchgeo.datasets.ChaBuD`.
        """
        assert temporal in ["pre", "post", "all"], f"Invalid temporal value: {temporal}. Must be one of ['pre', 'post', 'all']"
        self.temporal = temporal

        bands = kwargs.get('bands', ChaBuD.all_bands)
        band_indices = [ChaBuD.all_bands.index(b) for b in bands]
        mins = self.min[band_indices]
        maxs = self.max[band_indices]

        self.mean = mins
        self.std = maxs - mins

        super().__init__(ChaBuD, batch_size, num_workers, **kwargs)

    def setup(self, stage: str) -> None:
        """Set up datasets.

        Args:
            stage: Either 'fit', 'validate', 'test', or 'predict'.
        """
        if stage in ['fit', 'validate']:
            self.train_dataset = CustomChaBuDDataset(temporal=self.temporal, split="train", **self.kwargs)
            self.val_dataset = CustomChaBuDDataset(temporal=self.temporal, split="val", **self.kwargs)




In [ ]:
data_module = CustomChaBuDDataModule(temporal="post", download=True, batch_size=2, root="data/cha_bu_d", bands=["B02", "B03", "B04", "B08", "B11", "B12"])
data_module.prepare_data()
data_module.setup(stage="fit")

## Build a Custom Model for the task
Let's try with Prithvi V1 100M

In [ ]:
## build custom model for task

import terratorch
from terratorch.models import EncoderDecoderFactory
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datasets import HLSBands
# model_factory = EncoderDecoderFactory()

# build a segmentation model
# Parameters prefixed with backbone_ get passed to the backbone
# Parameters prefixed with decoder_ get passed to the decoder
# Parameters prefixed with head_ get passed to the head

model_args= dict(
    backbone = "prithvi_eo_v1_100",
    backbone_pretrained = True,
    backbone_bands = [
        HLSBands.BLUE,
        HLSBands.GREEN,
        HLSBands.RED,
        HLSBands.NIR_NARROW,
        HLSBands.SWIR_1,
        HLSBands.SWIR_2
    ],
    decoder = "FCNDecoder",
    decoder_channels = 256,
    decoder_num_convs=4,
    head_dropout = 0.1,
    num_classes = 2,
    necks=[dict(name="SelectIndices", indices=[-1]), dict(name="ReshapeTokensToImage")]

)

task = SemanticSegmentationTask(model_args,
                                model_factory="EncoderDecoderFactory",
                                freeze_backbone=True, freeze_decoder=False, optimizer="AdamW", lr=1e-3, class_weights=[0.53427446, 7.7940613 ])

In [ ]:
from lightning import Trainer

trainer = Trainer(
    accelerator="auto",
    log_every_n_steps=1,
    enable_progress_bar=True,
    enable_model_summary=True,
    max_epochs=5, check_val_every_n_epoch=15)

trainer.fit(task, data_module)

In [ ]:
trainer.validate(task, data_module)